This Notebook is used to download data from various Python-friendly sources.

# Berkeley Earth Temperature Data
https://berkeleyearth.org/data/
- [Global_TAVG_Gridded_5deg.nc](https://storage.googleapis.com/berkeley-earth-temperature-hr/global/gridded/Global_TAVG_Gridded_5deg.nc)

In [ ]:
# imports
from typing import Dict, List, Optional, Tuple
import os

import numpy as np
import pandas as pd
import xarray as xr
from matplotlib import pyplot as plt


In [ ]:
# constants
BASE_PATH = os.path.join('..', 'data')
# raw original dataset
DATA_PATH = os.path.join(BASE_PATH, 'raw', 'Global_TAVG_Gridded_5deg.nc')
# intermediate datasets
OUT_INTER_PATH1 = os.path.join(BASE_PATH, 'actual', 'temperature', 'temperature_annual-avg.csv')
OUT_INTER_PATH2 = os.path.join(BASE_PATH, 'actual', 'temperature', 'temperature_anual-avg_filtered.csv')
OUT_INTER_PATH3 = os.path.join(BASE_PATH, 'actual', 'temperature', 'temperature_baseline.csv')
# final dataset
OUT_PATH = os.path.join(BASE_PATH, "actual", "temperature", "temperature.csv")

In [ ]:
def filter_nan_temp(dataframe: pd.DataFrame) -> pd.DataFrame:
  return dataframe[dataframe['temperature'].notnull()].reset_index(drop=True)

In [ ]:
# load dataset
ds = xr.open_dataset(DATA_PATH, engine="netcdf4")
years = np.floor(ds.time.values).astype(int)
annual = (
    ds["temperature"]
    .assign_coords(year=("time", years))
    .groupby("year")
    .mean()
)
df = annual.to_dataframe().reset_index()
df_filtered = filter_nan_temp(df)

df_filtered.head()

In [ ]:
if False:
  df.to_csv(OUT_INTER_PATH1, index=True, index_label='id')
  df_filtered.to_csv(OUT_INTER_PATH2, index=True, index_label='id')

In [ ]:
year_start = 2020
year_end = 2025
baseline = annual.sel(
    year=slice(str(year_start), str(year_end))
).mean("year", skipna=True)

df_baseline = filter_nan_temp(baseline.to_dataframe().reset_index())
print(df_baseline.temperature.min())
print(df_baseline.temperature.max())
df_baseline.to_csv(OUT_INTER_PATH3, index=True, index_label="id")

## Generate Our Dataset

In [ ]:
# transform data into our internal structure
def compute_pain(temperature_value: float, min_temperature: float, max_temperature: float) -> float:
    temp_range = max_temperature - min_temperature
    temp_offset = min_temperature
    return (temperature_value - temp_offset) / temp_range

def normalize_temperature_dataset(dataframe: pd.DataFrame, category: str = "Temperature") -> pd.DataFrame:
    max_temp = dataframe['temperature'].max()
    min_temp = dataframe['temperature'].min()
    data: List[Dict] = []
    for _, row in dataframe.iterrows():
        lat = row['latitude']
        lon = row['longitude']
        temp = row['temperature']
        #value = 0.5 - 0.5 * np.exp(-(temp - temp_offset) / temp_range)
        #if lon < 0 and lat < 0:
        #    value = 1.0
        value = compute_pain(temp, min_temp, max_temp)
        
        data.append({
            'aggrId': None,
            'value': np.round(value, 5),
            'category': category,
            'lat': lat,
            'lng': lon,
        })

    return pd.DataFrame(data)

In [ ]:
df_baseline = pd.read_csv(OUT_INTER_PATH3, index_col='id')
df_temp = normalize_temperature_dataset(df_baseline)
df_temp.to_csv(OUT_PATH, index=True, index_label="id")

In [ ]:
_max_temp = df_baseline['temperature'].max()
_min_temp = df_baseline['temperature'].min()
_temp_range = _max_temp - _min_temp
_temp_offset = _min_temp

print(_max_temp)
print(_min_temp)
print(_temp_range)
print(_temp_offset)

## Visualize the Number of Missing Values
i.e., NaN values

In [ ]:
value_counts = df_filtered["year"].value_counts(sort=False).to_list()
value_start = 1850
plt.bar(x=range(value_start, value_start + len(value_counts)), height=value_counts)

## Visualize the Temperature Values

In [ ]:
dataset = pd.read_csv(OUT_PATH)

In [ ]:
print("min = ", dataset.value.min())
print("max = ", dataset.value.max())

In [ ]:
plt.plot(dataset.value)

In [ ]:
def my_plot(lat: float, lon: float):
  df_full = annual.to_dataframe().reset_index()
  #df_full = df_full[f_lat - step_size < df_full['latitude']]
  df_full = df_full[df_full['latitude'] == lat]
  #df_full = df_full[f_lon - step_size < df_full['longitude']]
  df_full = df_full[df_full['longitude'] == lon]
  df_full = filter_nan_temp(df_full)
  plt.plot(df_full.year, df_full.temperature)

In [ ]:
my_plot(47.5, 12.5)

# Source 2